# Dataset Keşifsel Analizi

Bu notebook dataset'lerin istatistiksel özelliklerini inceler.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Stil ayarları
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

In [ ]:
# Dataset'leri yükle
data_dir = Path('../../data/processed')

datasets = {}
for file in data_dir.glob('*.json'):
    with open(file, 'r', encoding='utf-8') as f:
        datasets[file.stem] = json.load(f)

print(f"Yüklenen dataset'ler: {list(datasets.keys())}")

In [ ]:
# Tüm dataset'i DataFrame'e çevir
all_data = datasets.get('all_dataset', [])
df = pd.DataFrame(all_data)

print(f"Toplam segment: {len(df)}")
print(f"\nKolonlar: {df.columns.tolist()}")
df.head()

## Temel İstatistikler

In [ ]:
# Kategori dağılımı
print("Kategori Dağılımı:")
print(df['category'].value_counts())

plt.figure(figsize=(10, 6))
df['category'].value_counts().plot(kind='bar', color='skyblue')
plt.title('Kategori Dağılımı')
plt.xlabel('Kategori')
plt.ylabel('Segment Sayısı')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Kaynak dağılımı
print("\nKaynak Dağılımı:")
print(df['source'].value_counts())

plt.figure(figsize=(10, 6))
df['source'].value_counts().plot(kind='barh', color='lightcoral')
plt.title('Dataset Kaynak Dağılımı')
plt.xlabel('Segment Sayısı')
plt.ylabel('Kaynak')
plt.tight_layout()
plt.show()

In [ ]:
# Uzunluk istatistikleri
print("\nUzunluk İstatistikleri:")
print(df['length'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['length'], bins=50, color='lightgreen', edgecolor='black')
axes[0].set_title('Metin Uzunluğu Dağılımı')
axes[0].set_xlabel('Karakter Sayısı')
axes[0].set_ylabel('Frekans')

# Box plot
df.boxplot(column='length', by='category', ax=axes[1])
axes[1].set_title('Kategoriye Göre Uzunluk Dağılımı')
axes[1].set_xlabel('Kategori')
axes[1].set_ylabel('Karakter Sayısı')

plt.tight_layout()
plt.show()

In [ ]:
# Placeholder analizi
if 'metadata' in df.columns:
    df['has_placeholder'] = df['metadata'].apply(lambda x: x.get('has_placeholder', False))
    
    print("\nPlaceholder İstatistikleri:")
    print(df['has_placeholder'].value_counts())
    
    plt.figure(figsize=(8, 6))
    df['has_placeholder'].value_counts().plot(kind='pie', autopct='%1.1f%%', 
                                               labels=['Placeholder Yok', 'Placeholder Var'])
    plt.title('Placeholder İçeren Segment Oranı')
    plt.ylabel('')
    plt.show()

## Kategori Bazlı Analiz

In [ ]:
# Kategoriye göre ortalama uzunluk
category_stats = df.groupby('category')['length'].agg(['mean', 'median', 'std', 'min', 'max'])
print("\nKategoriye Göre Uzunluk İstatistikleri:")
print(category_stats)

category_stats['mean'].plot(kind='bar', color='purple', alpha=0.7)
plt.title('Kategoriye Göre Ortalama Metin Uzunluğu')
plt.xlabel('Kategori')
plt.ylabel('Ortalama Karakter Sayısı')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Örnek metinler
print("\nÖrnek Metinler:")
print("\n" + "="*60)

for category in df['category'].unique():
    print(f"\n{category.upper()}:")
    samples = df[df['category'] == category].sample(min(3, len(df[df['category'] == category])))
    
    for idx, row in samples.iterrows():
        print(f"\n  EN: {row['source_text'][:100]}...")
        print(f"  TR: {row['target_text'][:100]}...")

## Sonuç

Dataset analizi tamamlandı. Bulgular:
- Toplam segment sayısı ve dağılımı
- Kategori bazlı özellikler
- Uzunluk istatistikleri
- Placeholder kullanımı